In [3]:
!pip install faker pandas
import random, string
import pandas as pd
from faker import Faker

fake = Faker()
random.seed(42)
Faker.seed(42)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
def rand_str(n, chars=string.ascii_letters + string.digits):
    return ''.join(random.choice(chars) for _ in range(n))

def make_email():
    t = random.choice([
        "Contact: {e}", "email me at {e}", "From: {e}", "{e}",
        "Send the report to {e} by Friday", "To: {e}", "User email: {e}"
    ])
    return t.format(e=fake.email()), "email"

def make_phone():
    t = random.choice([
        "Call me on {p}", "Phone: {p}", "{p}", "WhatsApp {p}",
        "Mobile number: {p}", "Reach me at {p} after 5pm"
    ])
    return t.format(p=fake.phone_number()), "phone"

def make_url_internal():
    t = random.choice([
        "http://192.168.{a}.{b}/admin",
        "http://10.0.{a}.{b}:8080/login",
        "https://internal.{c}.local/dashboard",
        "https://staging.{c}.com/api/v1/users",
        "http://localhost:{port}/debug",
        "https://jenkins.{c}.corp/job/deploy"
    ])
    c = fake.domain_word()
    return t.format(a=random.randint(0,255), b=random.randint(1,254),
                    c=c, port=random.randint(3000,9000)), "url_internal"

def make_credential():
    t = random.choice([
        "password={pw}", "DB_PASS = '{pw}'", "API_KEY=sk-{k}",
        "AWS_ACCESS_KEY_ID=AKIA{u}", "token: ghp_{g}",
        "Authorization: Bearer {j}", "secret_key: {k}",
        "login with user admin and pass {pw}", "PRIVATE_KEY={k}"
    ])
    text = t.format(
        pw=fake.password(length=random.randint(8, 16)),
        k=rand_str(32),
        u=rand_str(16, string.ascii_uppercase + string.digits),
        g=rand_str(36),
        j=rand_str(20) + "." + rand_str(40) + "." + rand_str(20)
    )
    return text, "credential"

def make_safe():
    # normal text plus "tricky" safe lines that look a bit like sensitive data
    kind = random.choice(["sentence", "sentence", "sentence", "public_url", "version", "misc"])
    if kind == "sentence":
        return fake.sentence(nb_words=random.randint(4, 14)), "safe"
    if kind == "public_url":
        return random.choice(["https://www.google.com", "https://github.com/" + fake.user_name(),
                              "https://www.wikipedia.org", "https://www.youtube.com/watch?v=" + rand_str(11)]), "safe"
    if kind == "version":
        return f"Version {random.randint(1,9)}.{random.randint(0,9)}.{random.randint(0,9)} released", "safe"
    if kind == "misc" and random.random() < 0.4:
        return random.choice([
            "Please call back later", "Phone settings", "Email notifications are off",
            "Forgot your password?", "Enter your password", "Contact support for help",
            "Call history cleared", "Your phone is up to date", "Check your email inbox",
            "Reset password link expired", "Token expires in 5 minutes",
        ]), "safe"    
    return random.choice([f"Order #{random.randint(10000,99999)} shipped",
                          f"Meeting at {random.randint(1,12)}:{random.choice(['00','30'])} pm",
                          f"Total: ${random.randint(5,500)}.{random.randint(0,99):02d}",
                          "File saved successfully", "Click here to continue", "Settings updated"]), "safe"

In [5]:
def make_url_internal():
    host = random.choice([
        f"192.168.{random.randint(0,255)}.{random.randint(1,254)}",
        f"10.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}",
        f"{random.choice(['internal','staging','dev','admin','jenkins','grafana','vpn'])}.{fake.domain_word()}.{random.choice(['local','corp','internal','lan'])}",
        "localhost",
    ])
    port = random.choice(["", f":{random.randint(3000,9999)}"])
    path = random.choice(["/admin", "/login", "/dashboard", "/api/v1/users", "/debug",
                          f"/{fake.word()}/{fake.word()}", f"/api/{fake.word()}?id={random.randint(1,9999)}",
                          f"/Reports/{fake.word().capitalize()}/index.html"])
    scheme = random.choice(["http://", "https://"])
    prefix = random.choice(["", "", "Open ", "URL: ", "Visit ", "Go to "])
    return f"{prefix}{scheme}{host}{port}{path}", "url_internal"

In [6]:
counts = {make_safe: 6000, make_email: 1000, make_phone: 1000,
          make_url_internal: 800, make_credential: 1200}

rows = []
for fn, n in counts.items():
    for _ in range(n):
        text, label = fn()
        rows.append({"text": text, "label": label, "source": "faker_synthetic"})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv("../data/synthetic.csv", index=False)

print(df.shape)
print(df["label"].value_counts())
df.sample(10)

(10000, 3)
label
safe            6000
credential      1200
email           1000
phone           1000
url_internal     800
Name: count, dtype: int64


,text,label,source
1931,Realize another high indicate small rate whole...,safe,faker_synthetic
9534,Another big treatment help rest shake research...,safe,faker_synthetic
8210,Medical sport knowledge performance very thous...,safe,faker_synthetic
4441,Send the report to amanda74@example.net by Friday,email,faker_synthetic
4349,Goal gas truth draw begin act including sell s...,safe,faker_synthetic
6890,Them sing vote feeling example recognize table...,safe,faker_synthetic
4137,email me at jamesbauer@example.net,email,faker_synthetic
5680,Total: $247.30,safe,faker_synthetic
7827,https://www.wikipedia.org,safe,faker_synthetic
9301,Call me on +1-990-650-2406x511,phone,faker_synthetic


In [7]:
import pandas as pd
s = pd.read_csv("../data/synthetic.csv")
u = s[s["label"] == "url_internal"]["text"]
print(u.str.len().describe().round(1))
print(u.sample(8, random_state=1).tolist())

count    800.0
mean      40.0
std        9.1
min       22.0
25%       33.0
50%       39.0
75%       46.2
max       71.0
Name: text, dtype: float64
['URL: https://internal.reeves.corp/Reports/While/index.html', 'Open https://192.168.60.174:8514/dashboard', 'https://192.168.121.138:6816/api/responsibility?id=7965', 'https://localhost/debug', 'Open https://localhost/api/v1/users', 'https://192.168.106.27:5110/together/kid', 'https://10.46.149.167/debug', 'https://192.168.13.210/api/set?id=6304']


In [8]:
df = pd.read_csv("../data/clean_dataset.csv")
print(df[(df["label"] == "safe") & df["text"].str.contains("http")]["text"].sample(5, random_state=1).tolist())

['https://github.com/murraymichelle', 'https://github.com/courtney10', 'https://www.youtube.com/watch?v=unDuvW4yrW8', 'https://github.com/anna38', 'https://www.youtube.com/watch?v=WLoJd1y0HA9']


# 01 - Build Synthetic Dataset (Faker)

## Purpose
Create the part of the dataset that no public source provides: credentials, internal
URLs, and clean examples of emails and phones. All values are fake, so there is no
privacy risk.

## Input / Output
- **Input:** none (data is generated with code)
- **Output:** `data/synthetic.csv` (columns: text, label, source)

## What we did
1. Installed and imported Faker and fixed random seeds (42) so results can be repeated.
2. Wrote one generator function per class:
   - `make_email`, `make_phone`: sentence templates around Faker emails and phones
   - `make_url_internal`: private IPs, localhost, `.local/.corp` hosts, ports, paths
   - `make_credential`: passwords, API keys, AWS keys, GitHub tokens, bearer tokens
   - `make_safe`: normal sentences plus "tricky" safe lines (public URLs, version
     numbers, order numbers) so the model cannot just look for digits or "http"
3. Generated 10,000 rows: safe 6000, credential 1200, email 1000, phone 1000,
   url_internal 800, then shuffled and saved.
4. Improved `make_url_internal` after EDA showed only 6 templates (see below).

## Key concepts learned
- **Synthetic data** lets us create classes that do not exist in public datasets.
- **Random seeds** make a dataset reproducible for the whole team.
- **Tricky negatives** (safe text that looks sensitive) make the model learn real
  patterns instead of shortcuts.
- A dataset flows through a chain of files. Changing this notebook has no effect until
  notebooks 03 and 04 are re-run.

## Problems and fixes
- First URL generator used only 6 templates, so the model could memorize them. We added
  varied hosts, ports, paths, prefixes and mixed-case paths.
- Redefining a function does not update the `counts` dictionary that stores the old
  function. We restart the kernel and use Run All.

## Brief requirement covered
Data source and description, relevance to the problem, dataset building.